# TTC Station Reliability

Which TTC stations are actually unreliable, and which just look bad because
they are busy? Delay counts are divided by scheduled trips to get a rate.

Run top to bottom. Tables persist in `ttc.duckdb`, so after a kernel restart
only the connection cell needs re-running.

## Setup — file inventory

24 files across `delay/`, `ridership/` and `schedules/`, mixed xlsx / csv / txt.

In [80]:
# Verifying data path

import os
import glob
import pandas as pd

paths = glob.glob("data/raw/**/*", recursive=True)

files = []
for item in paths:
    if os.path.isfile(item):
        files.append(item)

length = len(files)

print(files)
print(f"\n total files: {length}")

['data/raw\\delay\\TTC Subway Delay Data since 2025.csv', 'data/raw\\delay\\ttc-subway-delay-data-2018.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2019.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2020.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2021.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2022.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2023.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2024.xlsx', 'data/raw\\delay\\ttc-subway-delay-jan-2014-april-2017.xlsx', 'data/raw\\delay\\ttc-subway-delay-may-december-2017.xlsx', 'data/raw\\ridership\\1985-2019 Analysis of ridership.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2012-2013.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2014.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2015.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2016.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2017.xlsx', 'data/raw\\schedules\\agency.txt', 'data/raw\\schedules\\calendar.txt', 'data/raw\\schedules\\c

## Load GTFS

The schedule feed is the denominator, how many trains are scheduled to stop
at each station. 

This is chosen over ridership data as the datasets ends in 2019 which would not allow us to factor in COVID 19 disruptions and potential post COVID changes that might affect present day opertaions.

`route_type = 1` is subway, 
0 and 3 are persumably streetcarts and buses respectively

In [81]:
# Verifying routes 

import duckdb

con = duckdb.connect("ttc.duckdb")


con.sql("CREATE OR REPLACE TABLE routes AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/routes.txt')")


con.sql("SELECT COUNT(route_id) FROM routes GROUP BY route_type").df()



,count(route_id)
0,20
1,3
2,210


In [82]:
con.sql("SELECT route_id, route_short_name, route_long_name " \
"  FROM routes " \
"WHERE route_type = 1").df()

,route_id,route_short_name,route_long_name
0,1,1,Line 1 (Yonge-University)
1,2,2,Line 2 (Bloor - Danforth)
2,4,4,Line 4 (Sheppard)


### Load the remaining GTFS tables

`stop_times` is the bridge: it carries both `trip_id` and `stop_id`.

In [83]:
con.sql("CREATE OR REPLACE TABLE stops AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/stops.txt')")

con.sql("CREATE OR REPLACE TABLE stop_times AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/stop_times.txt')")

con.sql("CREATE OR REPLACE TABLE trips AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/trips.txt')")

## Denominator — scheduled trips per station

Filter `stop_times` to subway trips only, cutting 4.2M rows to ~150k.

In [84]:
# filter in only stop time data related to subway lines

con.sql("""
    CREATE OR REPLACE TABLE subway_stop_times AS
    SELECT stop_times.*, route_id FROM trips
    JOIN stop_times 
        ON trips.trip_id = stop_times.trip_id
    WHERE trips.route_id IN [1,2,4]
""")

con.sql("""
    SELECT * FROM subway_stop_times LIMIT 5
""").df()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,route_id
0,50659966,5:45:18,5:45:18,14945,1,None,0,0,NaN,1
1,50659966,5:47:44,5:47:44,15664,2,None,0,0,1.4442,1
2,50659966,5:50:22,5:50:22,15659,3,None,0,0,3.3901,1
3,50659966,5:52:34,5:52:34,15666,4,None,0,0,4.7396,1
4,50659966,5:54:26,5:54:26,15656,5,None,0,0,5.5524,1


### `parent_station` is null for TTC, so stations group on `stop_name`

In [85]:
con.sql("""
    SELECT * FROM stops LIMIT 5
""").df()

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
0,662,662,Danforth Rd at Kennedy Rd,None,43.714379,-79.260939,None,None,None,None,None,1
1,929,929,Davenport Rd at Bedford Rd,None,43.674448,-79.399659,None,None,None,None,None,1
2,940,940,Davenport Rd at Dupont St,None,43.675511,-79.401938,None,None,None,None,None,2
3,1871,1871,Davisville Ave at Cleveland St,None,43.702088,-79.378112,None,None,None,None,None,1
4,11700,11700,Disco Rd at Attwell Dr,None,43.701362,-79.594843,None,None,None,None,None,1


In [86]:
pd.set_option('display.max_rows', 20)

con.sql("""
    SELECT DISTINCT(route_type) FROM routes
""").df()

,route_type
0,0
1,1
2,3


### Merge Bloor-Yonge and normalise names

GTFS logs Bloor-Yonge as two stations — "Bloor" for the Line 2 platforms and
"Yonge" for Line 1. Merged here, giving 70 stations. Names are then uppercased
and stripped of " Station" to match the delay data's vocabulary.

Sanity check: the four interchanges show ~4,300 trips, ordinary stations
~2,100, Line 4 stops ~1,760.

In [ ]:
## fixing name duplicate i.e Bloor-Yonge

pd.set_option('display.max_rows', 80)

con.sql("""
    CREATE OR REPLACE TABLE station_trips AS
        SELECT 
            route_id AS line,
            SPLIT_PART(stop_name, ' -', 1) AS stations, 
            COUNT(*) AS scheduled_trips
        FROM stops
        JOIN subway_stop_times
            ON subway_stop_times.stop_id = stops.stop_id
        GROUP BY stations, line
        ORDER BY stations ASC
""")

con.sql("""
    CREATE OR REPLACE TABLE station_trips_clean AS
        WITH 
        s1 AS (
            SELECT * REPLACE (SPLIT_PART(stations, ' Station' , 1) AS stations) FROM station_trips),
        s2 AS (
            SELECT * REPLACE (UPPER(stations) AS stations) FROM s1),
        s3 AS (
            SELECT * RENAME (stations AS station) FROM s2),
        s4 AS (
            SELECT * REPLACE (
                CASE WHEN station IN ('BLOOR', 'YONGE') THEN 'BLOOR-YONGE'
                ELSE station END AS station)
            FROM s3)
    SELECT * FROM s4
""")

con.sql("""SELECT * FROM station_trips_clean""").df()


ParserException: Parser Error: syntax error at or near "THEN"

LINE 12:                 CASE station IN ('BLOOR', 'YONGE') THEN 'BLOOR-YONGE'
                                                            ^

In [ ]:
pd.set_option('display.max_rows', 20)

con.sql("""
    SELECT station, COUNT(station) AS n_lines
    FROM station_trips_clean
    GROUP BY station
""").df()

,station,n_lines
0,BAY,1
1,CHESTER,1
2,ISLINGTON,1
3,KEELE,1
4,MAIN STREET,1
5,SUMMERHILL,1
6,WILSON,1
7,COLLEGE,1
8,DAVISVILLE,1
9,EGLINTON,1


## Load delay data

10 files, Jan 2014 – Jun 2026. Column schema is stable across all of them;
the csv adds an `_id` column from CKAN.

In [88]:
## Verifying delay datas

paths = glob.glob("data/raw/delay/**")

for f in paths:
    if os.path.splitext(f)[1].lower() == '.xlsx':
        df = pd.read_excel(f, nrows=0)
    else:
        df = pd.read_csv(f, nrows=0)
    print(f, df.columns.tolist())

path = "data/raw/delay/ttc-subway-delay-jan-2014-april-2017.xlsx"
sheets = pd.read_excel(path, sheet_name=None)
print(sheets.keys())

print(sheets['Incidents']['Date'].min())
print(sheets['Incidents']['Date'].max())

data/raw/delay\TTC Subway Delay Data since 2025.csv ['_id', 'Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2018.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2019.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2020.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2021.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2022.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2023.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehi

In [89]:
paths = glob.glob("data/raw/delay/**")

dfs = []

for f in paths:
    if os.path.splitext(f)[1].lower() == '.xlsx':
        dfs.extend(pd.read_excel(f, sheet_name=None).values())
    else:
        dfs.append(pd.read_csv(f))

all_delays = pd.concat(dfs, ignore_index=True)

## all_delays.shape

(all_delays.dtypes.to_string())

## print(all_delays['Date'].max())
## print(all_delays['Date'].min())

'_id          float64\nDate          object\nTime             str\nDay              str\nStation          str\nCode             str\nMin Delay      int64\nMin Gap        int64\nBound            str\nLine             str\nVehicle        int64'

### Station name concentration

1,853 distinct station values against a true count of 70. The top 70 cover
~90% of rows and the top 150 ~98%, so the ~1,700-value tail is almost all
singletons — which is what makes exclusion defensible rather than mapping.

In [90]:
pd.set_option('display.max_rows', 100)

con.sql("""
CREATE OR REPLACE TABLE raw_delays AS
SELECT * FROM all_delays 
""")

n = con.sql("SELECT COUNT(DISTINCT Station) AS n FROM raw_delays").fetchone()[0]

print(f"distinct stations: {n}\n")

con.sql("""
WITH ranked AS (
    SELECT
        Station, 
        COUNT(Station) AS count,
        count / SUM(COUNT(*)) OVER () AS ratio,
        SUM(COUNT(*)) OVER (ORDER BY COUNT(*) DESC) as r_total,
        r_total / SUM(COUNT(*)) OVER () AS r_ratio,
        ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS r_num
    FROM raw_delays
    GROUP BY Station
    ORDER BY Count DESC
)
SELECT * FROM ranked WHERE r_num IN (70,150)
""").df()



distinct stations: 2178



,Station,count,ratio,r_total,r_ratio,r_num
0,RUNNYMEDE STATION,1451,0.005517,236483.0,0.899096,70
1,KIPLING STATION (APPRO,24,0.000091,258660.0,0.983412,150


### Raw delay table

All 186,916 rows as concatenated, before any cleaning.

In [91]:
con.sql("""
SELECT * FROM all_delays
""").df()

,_id,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
0,1.0,2025-01-01,02:10,Wednesday,BATHURST STATION,MUSAN,5,9,E,BD,5227
1,2.0,2025-01-01,02:30,Wednesday,DUNDAS STATION,MUIRS,0,0,NaN,YU,0
2,3.0,2025-01-01,02:32,Wednesday,BROADVIEW STATION,PUMST,0,0,E,BD,0
3,4.0,2025-01-01,02:58,Wednesday,KEELE STATION,EUSC,0,0,W,BD,5293
4,5.0,2025-01-01,02:58,Wednesday,COXWELL STATION,SUAE,0,0,NaN,BD,0
5,6.0,2025-01-01,07:59,Wednesday,DONLANDS STATION,TUNOA,5,10,W,BD,5000
6,7.0,2025-01-01,08:13,Wednesday,BLOOR STATION,PUTR,9,0,S,YU,5836
7,8.0,2025-01-01,08:15,Wednesday,BLOOR STATION,PUTR,5,0,N,YU,5706
8,9.0,2025-01-01,09:14,Wednesday,FINCH STATION,SUDP,0,0,S,YU,5596
9,10.0,2025-01-01,09:22,Wednesday,BATHURST STATION,SUUT,0,0,E,BD,5312


## Clean layer

Each CTE applies one rule. Order matters:

- **s1** drops Line 3 (SRT) — closed 2023, absent from GTFS, no denominator.
  Must come before the suffix strip or Kennedy SRT merges into Kennedy.
- **s3** splits at `' STATION'`, which also removes trailing descriptors
  like `'ROYAL YORK STATION (AP'`.
- **s10** anchors `DUNDAS` → `TMU`, since `DUNDAS WEST` is a different station.
- **s19/s20** reconcile Date, which parses as a timestamp from xlsx and as
  text from the csv, then combine it with Time.
- **s21** is the exclusion step: the join drops yards, hostlers, carhouses,
  wyes, portals, two-station segments and line-level records — ~15k rows (10%),
  none of which can be attributed to a single station.

In [92]:
pd.set_option('display.max_rows', 25)

con.sql("""
CREATE OR REPLACE TABLE clean_delays_all AS 
    WITH 
    s1 AS (
        SELECT * FROM raw_delays WHERE Line IS DISTINCT FROM 'SRT'),
    s1a AS (
        SELECT * FROM s1 WHERE NOT station LIKE '% TO %'),
    s2 AS (
        SELECT * REPLACE (SPLIT_PART(Station, ' STATION', 1) AS Station) FROM s1a),
    s3 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, ' (BD|YU|YUS|SRT)$', '') AS Station) FROM s2),
    s4 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, 'SHEPPARDSTATION$', 'SHEPPARD-YONGE') AS Station) FROM s3),
    s5 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, 'BLOOR YONGE', 'BLOOR-YONGE') AS Station) FROM s4),
    s6 AS (
        SELECT * RENAME (Station AS station) FROM s5),
    s7 AS (
        SELECT * REPLACE (REPLACE(station, '.', '') AS station) FROM s6),
    s8 AS (
        SELECT * REPLACE (REPLACE(station, 'EGLINTON WEST', 'CEDARVALE') AS station) FROM s7),
    s9 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^DUNDAS$', 'TMU') AS station) FROM s8),
    s11 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^BLOOR$', 'BLOOR-YONGE') AS station) FROM s9),
    s12 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^YONGE$', 'BLOOR-YONGE') AS station) FROM s11),
    s13 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^SHEPPARD$', 'SHEPPARD-YONGE') AS station) FROM s12),
    s14 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^VAUGHAN MC$', 'VAUGHAN METROPOLITAN CENTRE') AS station) FROM s13),
    s15 AS (
        SELECT * REPLACE (REPLACE(station, 'CTR', 'CENTRE') AS station) FROM s14),
    s16 AS (
        SELECT * REPLACE (REPLACE(station, 'VMC', 'VAUGHAN METROPOLITAN CENTRE') AS station) FROM s15),
    s17 AS (
        SELECT * REPLACE (REPLACE(station, 'YONGE SHP', 'SHEPPARD-YONGE') AS station) FROM s16),
    s18 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, ' STATIO$', '') AS station) FROM s17),
    s19 AS (
        SELECT * EXCLUDE(Date), SPLIT_PART(Date, ' ', 1) AS Date_normalized FROM s18),
    s20 AS (
        SELECT *, 
                CASE Line 
                    WHEN 'YUS' THEN 1
                    WHEN 'YU' THEN 1
                    WHEN 'BD' THEN 2
                    WHEN 'SHP' THEN 4
                    WHEN 'SHEP' THEN 4
                END AS route
            FROM s19),        
    s21 AS (
        SELECT * EXCLUDE(_id, Time, Date_normalized), STRPTIME(Date_normalized || ' ' || Time, '%Y-%m-%d %H:%M') AS Timestamp FROM s20)
SELECT * FROM s21
""")

con.sql("""
    CREATE OR REPLACE TABLE clean_delays AS 
        SELECT * EXCLUDE(t1.station, scheduled_trips, Line) FROM clean_delays_all t1
        LEFT JOIN station_trips_clean t2
            ON t1.station = t2.station AND t1.route = t2.line
        WHERE t2.station IS NOT NULL
""")

result = con.sql("""
    SELECT * FROM clean_delays
""").df()

result



,Day,Code,Min Delay,Min Gap,Bound,Vehicle,route,Timestamp,station
0,Wednesday,MUSC,0,0,S,5711,1,2017-04-26 07:51:00,LAWRENCE WEST
1,Wednesday,TUSC,0,0,E,5317,2,2017-04-26 07:57:00,OLD MILL
2,Wednesday,TUSC,0,0,W,6161,4,2017-04-26 08:04:00,LESLIE
3,Wednesday,TUSUP,3,5,E,5000,2,2017-04-26 08:20:00,DUNDAS WEST
4,Wednesday,TUSC,4,6,E,5243,2,2017-04-26 08:22:00,ISLINGTON
5,Wednesday,MUSC,0,0,S,5066,1,2017-04-26 09:13:00,FINCH
6,Wednesday,SUUT,22,25,S,5491,1,2017-04-26 10:27:00,TMU
7,Wednesday,MUIS,0,0,NaN,0,2,2017-04-26 11:45:00,BATHURST
8,Wednesday,MUSC,0,0,W,6171,4,2017-04-26 11:59:00,DON MILLS
9,Wednesday,TUMVS,0,0,S,6026,1,2017-04-26 12:02:00,SHEPPARD-YONGE


In [93]:
pd.set_option('display.max_rows', 25)

con.sql("""
    WITH table1 AS (
        SELECT * FROM clean_delays_all t1
        JOIN ( SELECT DISTINCT station FROM station_trips_clean) t2
            ON t1.station = t2.station
        WHERE t2.station IS NOT NULL
    )

    SELECT t1.station, route, COUNT(t1.station) AS count FROM table1 t1
    LEFT JOIN station_trips_clean t2
        ON t1.route = t2.Line AND t1.station = t2.station
    WHERE t2.Line IS NULL
    GROUP BY t1.station, route
    ORDER BY count DESC
""").df()

,station,route,count
0,WARDEN,1,120
1,WARDEN,<NA>,46
2,KENNEDY,<NA>,36
3,KENNEDY,1,28
4,FINCH,<NA>,27
5,SPADINA,<NA>,25
6,KIPLING,1,22
7,KIPLING,<NA>,20
8,BROADVIEW,<NA>,14
9,QUEEN,<NA>,13


### After cleaning

In [94]:
pd.set_option('display.max_rows', 100)

con.sql("""
WITH ranked AS (
    SELECT
        Station, 
        COUNT(Station) AS count,
        count / SUM(COUNT(*)) OVER () AS ratio,
        SUM(COUNT(*)) OVER (ORDER BY COUNT(*) DESC) as r_total,
        r_total / SUM(COUNT(*)) OVER () AS r_ratio,
        ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS r_num
    FROM clean_delays_all
    GROUP BY Station
    ORDER BY Count DESC
)
SELECT * FROM ranked WHERE r_num IN (70,150)
""").df()

,station,count,ratio,r_total,r_ratio,r_num
0,HIGHWAY 407,1018,0.004006,245295.0,0.965189,70
1,WILSON GARAGE,8,0.000031,252992.0,0.995475,150


### Every row parsed — no null timestamps

In [95]:
con.sql("""
SELECT COUNT(*) FROM clean_delays
WHERE Timestamp IS NULL
""").df()

,count_star()
0,0


### `clean_delays` after all rules

In [96]:
pd.set_option('display.min_rows', 5)

con.sql("""
SELECT *
FROM clean_delays
""").df()

,Day,Code,Min Delay,Min Gap,Bound,Vehicle,route,Timestamp,station
0,Wednesday,MUSC,0,0,S,5711,1,2017-04-26 07:51:00,LAWRENCE WEST
1,Wednesday,TUSC,0,0,E,5317,2,2017-04-26 07:57:00,OLD MILL
...,...,...,...,...,...,...,...,...,...
224679,Friday,MUIRS,0,0,N,0,1,2022-07-22 10:19:00,WELLESLEY
224680,Friday,TUNIP,4,8,W,5114,2,2022-07-22 10:46:00,KENNEDY


## Exclusion checks

Segments name two stations, so they have no single denominator. Under 1% of
rows, too small to bias the ranking.

In [97]:
pd.set_option('display.min_rows', 10)
pd.set_option('display.max_rows', 10)

con.sql("""
SELECT t1.station, route, COUNT(*) AS count
FROM clean_delays_all AS t1 
LEFT JOIN station_trips_clean t2 
    ON t1.station = t2.station
WHERE t2.station IS NULL
GROUP BY t1.station, route 
ORDER BY count DESC
""").df()




,station,route,count
0,BLOOR-YONGE,1,7901
1,BLOOR-YONGE,2,4827
2,YONGE UNIVERSITY LINE,1,3210
3,BLOOR DANFORTH SUBWAY,2,2564
4,YONGE-UNIVERSITY AND B,<NA>,1927
...,...,...,...
1030,2233 SHEPPARD WEST,1,1
1031,KIPLING & UNION,<NA>,1
1032,ST GEORGE - LAWRENCE W,1,1
1033,BLOOR / DANFORTH LINE,2,1


### Delay rows per matched station

Coverage check: every one of the 70 GTFS stations has delay rows.

In [98]:
con.sql("""
SELECT t2.station, COUNT(*) AS count
FROM clean_delays AS t1 
RIGHT JOIN station_trips_clean t2 
    ON t1.station = t2.station
WHERE t1.station IS NOT NULL
GROUP BY t2.station
ORDER BY station ASC
""").df()


,station,count
0,BATHURST,2707
1,BAY,1945
2,BAYVIEW,1176
3,BESSARION,693
4,BROADVIEW,3044
...,...,...
64,WILSON,5642
65,WOODBINE,2562
66,YORK MILLS,3582
67,YORK UNIVERSITY,650


### Rows with no matching station

In [99]:
con.sql("""
SELECT COUNT(*) AS count
FROM clean_delays_all AS t1 
LEFT JOIN station_trips_clean t2 
    ON t1.station = t2.station
WHERE t2.station IS NULL
""").df()

,count
0,28418


### Zero-delay rows

65% of rows have `Min Delay = 0` — logged incidents with no measurable delay.
Nearly all also have zero gap, meaning no service impact, so they are excluded
from the metrics below.

In [100]:
pd.set_option('display.min_rows', 30)
pd.set_option('display.max_rows', 30)

con.sql("""
SELECT COUNT(*) FROM clean_delays 
WHERE "Min Delay" > 0
""").df()

,count_star()
0,81801


In [101]:
con.sql("""
SELECT COUNT(*) FROM clean_delays 
WHERE "Min Gap" > 0
""").df()

,count_star()
0,79348


## Analysis — full range

`delay_rate` = incidents / scheduled trips. Also computes total and average
delay minutes, plus p50 and p95 for severity.

In [122]:
## raw unreliability number

pd.set_option('display.min_rows', 40)
pd.set_option('display.max_rows', 40)

con.sql("""
CREATE OR REPLACE TABLE station_unreliability AS 
    WITH 
    s1 AS (
        SELECT station, 
            COUNT("Min Delay") AS delays, 
            SUM("Min Delay") AS total_delay, 
            AVG("Min Delay") AS avg_delay,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "Min Delay") AS p50,
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "Min Delay") AS p95,
            route
        FROM clean_delays 
        WHERE "Min Delay" > 0
        GROUP BY station, route),
    s2 AS (
        SELECT 
            t1.station, 
            delays / scheduled_trips AS delay_rate, 
            delays, total_delay, avg_delay,
            p50,
            p95,
            route
        FROM s1 t1 
        JOIN station_trips_clean t2 
            ON t1.station = t2.station)
    SELECT *, 
        RANK() OVER (ORDER BY delay_rate DESC) AS adjusted_rank, 
        RANK() OVER (ORDER BY delays DESC) AS raw_rank,
        raw_rank - adjusted_rank AS delta
    FROM s2
""")

### Extreme delays

A handful of multi-hour records. The Jan 2026 and Feb 2025 clusters span
multiple lines on consecutive days, so these are real system-wide disruptions
rather than entry errors. Retained — median and p95 are used instead of the
mean so they do not distort the ranking.

In [123]:
con.sql("""
SELECT station, route, "Min Delay", Timestamp FROM clean_delays
WHERE "Min Delay" > 400
""").df()

,station,route,Min Delay,Timestamp
0,MUSEUM,1,491,2024-03-01 06:08:00
1,DUNDAS WEST,2,424,2014-09-30 07:55:00
2,COLLEGE,1,452,2015-03-24 05:59:00
3,JANE,2,575,2016-05-19 16:35:00
4,EGLINTON,1,807,2025-02-16 09:18:00
5,SHEPPARD WEST,1,900,2025-02-16 11:03:00
6,VICTORIA PARK,2,661,2026-01-25 15:21:00
7,GLENCAIRN,1,622,2026-01-25 16:12:00
8,KIPLING,2,441,2026-01-25 18:32:00
9,WOODBINE,2,827,2026-01-26 05:50:00


### Data range Verification

In [104]:
con.sql("""
SELECT MAX(Timestamp), MIN(Timestamp) FROM clean_delays
""").df()

,"max(""Timestamp"")","min(""Timestamp"")"
0,2026-06-30 23:55:00,2014-01-01 00:21:00


## Analysis — 2018 onward

The Vaughan extension opened December 2017, so its six stations have delays
for only part of the full window while carrying the same denominator. Their
rates are understated. This version is the primary result.

In [125]:
## with 2017 line 1 extension sensitivity consideration

pd.set_option('display.min_rows', 40)
pd.set_option('display.max_rows', 40)

con.sql("""
CREATE OR REPLACE TABLE station_unreliability_2018 AS 
    WITH 
    s1 AS (
        SELECT station, 
            COUNT("Min Delay") AS delays, 
            SUM("Min Delay") AS total_delay, 
            AVG("Min Delay") AS avg_delay,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "Min Delay") AS p50,
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "Min Delay") AS p95,
            route
        FROM clean_delays 
        WHERE "Min Delay" > 0 AND Timestamp >= '2018-01-01'
        GROUP BY station, route),
    s2 AS (
        SELECT 
            t1.station, 
            delays / scheduled_trips AS delay_rate, 
            delays, total_delay, avg_delay,
            p50,
            p95,
            route
        FROM s1 t1 
        JOIN station_trips_clean t2 
            ON t1.station = t2.station)
    SELECT *, 
        RANK() OVER (ORDER BY delay_rate DESC) AS adjusted_rank, 
        RANK() OVER (ORDER BY delays DESC) AS raw_rank,
        raw_rank - adjusted_rank AS delta
    FROM s2
""")

## GTFS service calendar

Loaded but not currently used — the denominator counts trips defined in
the feed without weighting them by how many days each service actually runs.

In [126]:
con.sql("""
CREATE OR REPLACE TABLE calendar AS
    SELECT * FROM read_csv_auto('data/raw/schedules/calendar.txt')
""")

con.sql("""
SELECT * FROM calendar
""").df()

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,1,1,1,1,1,1,0,0,20260726,20260905
1,2,0,0,0,0,0,1,0,20260726,20260905
2,3,0,0,0,0,0,0,1,20260726,20260905
3,4,0,0,0,0,0,0,0,20260726,20260905
4,501,0,0,0,0,0,0,0,20260726,20260905
5,6701,0,0,0,0,0,0,0,20260726,20260905
6,4401,0,0,0,0,0,0,0,20260726,20260905
7,4501,0,0,0,0,0,0,0,20260726,20260905
8,7001,0,0,0,0,0,0,0,20260726,20260905
9,6702,0,0,0,0,0,0,0,20260726,20260905


### Sensitivity: full range vs 2018

Positive `adjusted_diff` means the station ranked better in the full-range
version than it should have. Highway 407 moves 21 places, Finch West 16 —
confirming the bias.

In [129]:
con.sql("""
SELECT 
    t1.station, 
    t1.raw_rank - t2.raw_rank AS raw_diff,
    t1.adjusted_rank - t2.adjusted_rank AS adjusted_diff,
    t1.route
FROM station_unreliability t1 
JOIN station_unreliability_2018 t2
    ON t1.station = t2.station
ORDER BY adjusted_diff DESC
""").df()

## the more positive the worst the delay

,station,raw_diff,adjusted_diff,route
0,HIGHWAY 407,17,18,1
1,SHEPPARD-YONGE,1,17,1
2,SHEPPARD-YONGE,-2,14,4
3,FINCH WEST,12,13,1
4,ST GEORGE,6,8,2
5,SPADINA,6,7,1
6,ST GEORGE,6,7,2
7,ROSEDALE,6,7,1
8,ST PATRICK,6,6,1
9,SHEPPARD-YONGE,-1,6,1


## Top 10 — adjusted rank, full range

In [132]:
con.sql("""
SELECT station, route, adjusted_rank FROM station_unreliability
ORDER BY adjusted_rank ASC
LIMIT 10
""").df()


,station,route,adjusted_rank
0,KENNEDY,2,1
1,FINCH,1,2
2,KIPLING,2,3
3,EGLINTON,1,4
4,VAUGHAN METROPOLITAN CENTRE,1,5
5,WILSON,1,6
6,SHEPPARD WEST,1,7
7,DAVISVILLE,1,8
8,COXWELL,2,9
9,ST GEORGE,1,10


## Top 10 — adjusted rank, 2018 onward

In [133]:
con.sql("""
SELECT station, route, adjusted_rank FROM station_unreliability_2018
ORDER BY adjusted_rank ASC
LIMIT 10
""").df()

,station,route,adjusted_rank
0,VAUGHAN METROPOLITAN CENTRE,1,1
1,FINCH,1,2
2,KENNEDY,2,3
3,KIPLING,2,4
4,EGLINTON,1,5
5,WILSON,1,6
6,SHEPPARD WEST,1,7
7,COXWELL,2,8
8,DAVISVILLE,1,9
9,ST GEORGE,1,10


## Top 10 — raw count, full range

In [135]:
con.sql("""
SELECT station, route, raw_rank FROM station_unreliability
ORDER BY raw_rank ASC
LIMIT 10
""").df()

,station,route,raw_rank
0,KENNEDY,2,1
1,FINCH,1,2
2,KIPLING,2,3
3,EGLINTON,1,4
4,VAUGHAN METROPOLITAN CENTRE,1,5
5,WILSON,1,6
6,SHEPPARD WEST,1,7
7,DAVISVILLE,1,8
8,COXWELL,2,9
9,ST GEORGE,1,10


## Top 10 — raw count, 2018 onward

Comparing this against the adjusted list gives the headline: the stations that
drop out are all interchanges, which carry roughly double the scheduled trips.

In [136]:
con.sql("""
SELECT station, route, raw_rank FROM station_unreliability_2018
ORDER BY raw_rank ASC
LIMIT 10
""").df()

,station,route,raw_rank
0,VAUGHAN METROPOLITAN CENTRE,1,1
1,FINCH,1,2
2,KENNEDY,2,3
3,KIPLING,2,4
4,EGLINTON,1,5
5,WILSON,1,6
6,SHEPPARD WEST,1,7
7,DAVISVILLE,1,8
8,COXWELL,2,9
9,ST GEORGE,1,10


### Denominator recap

In [137]:
con.sql("""
    SELECT * FROM station_trips_clean
""").df()

,line,station,scheduled_trips
0,2,BATHURST,2137
1,2,BAY,2136
2,4,BAYVIEW,1755
3,4,BESSARION,1758
4,1,BLOOR,2196
5,2,BROADVIEW,2133
6,2,CASTLE FRANK,2133
7,1,CEDARVALE,2181
8,2,CHESTER,2133
9,2,CHRISTIE,2139


## Confidence intervals

A parametric bootstrap. Delay counts should follow a Poisson distribution, and
a Poisson's variance equals its mean, so the observed count is the only
parameter needed. Each station gets 5,000 simulated counts, converted to rates,
with the 2.5th and 97.5th percentiles taken as a 95% interval.

`worse_than_baseline` checks whether even the low end of a station's interval
still sits above the network average rate.

One limitation. Poisson assumes events happen independently, but delays
clusters, since a single incident often generates several records. That means
the real counts are overdispersed and these intervals come out narrower than
they should be. A negative binomial model would fix this.

In [139]:
df = con.sql("""
    SELECT t1.station, route, delays, scheduled_trips
    FROM station_unreliability_2018 t1 
    JOIN station_trips_clean t2 ON t1.station = t2.station
""").df()

import numpy as np
rng = np.random.default_rng(1)

sims = rng.poisson(df['delays'].values,size=(5000, len(df)))

rates = sims / df['scheduled_trips'].values

df['ci_low'], df['ci_high'] = np.percentile(rates, [2.5, 97.5], axis=0)
df['rate'] = df['delays'] / df['scheduled_trips']

baseline = df['delays'].sum() / df['scheduled_trips'].sum()
df['worse_than_baseline'] = df['ci_low'] > baseline

df.sort_values('rate', ascending=False).head(15)

,station,route,delays,scheduled_trips,ci_low,ci_high,rate,worse_than_baseline
0,VAUGHAN METROPOLITAN CENTRE,1,3029,2154,1.358867,1.455896,1.406221,True
1,FINCH,1,2922,2224,1.266187,1.361511,1.313849,True
2,KENNEDY,2,2745,2158,1.225672,1.319289,1.272011,True
3,KIPLING,2,2477,2155,1.104872,1.195360,1.149420,True
4,EGLINTON,1,2459,2223,1.061628,1.151608,1.106163,True
5,WILSON,1,2222,2181,0.977075,1.062357,1.018799,True
6,SHEPPARD WEST,1,1583,2154,0.700557,0.771135,0.734912,True
8,COXWELL,2,1461,2158,0.642725,0.710843,0.677016,True
7,DAVISVILLE,1,1474,2190,0.639269,0.707306,0.673059,True
9,ST GEORGE,1,1345,2137,0.596151,0.663091,0.629387,True


## Export

`station_data_final` joins the 2018 metrics to the bootstrap intervals.
`rank_comparison` is the same ranks in long format, which is the shape
Tableau needs for a slope chart.

In [114]:
con.sql("""
    CREATE OR REPLACE TABLE station_ci AS
        SELECT * FROM df
""")

con.sql("""
    CREATE OR REPLACE TABLE station_data_final AS
        SELECT * EXCLUDE(t2.station, t2.delays, rate) FROM station_unreliability_2018 t1 
        JOIN station_ci t2 
            ON t1.station = t2.station 
""")

con.sql("""
    SELECT * FROM station_data_final
""").df()


con.sql("""
    COPY station_data_final TO 'output/station_final.csv' (HEADER, DELIMITER ',')
""")


In [115]:
con.sql("""
    SELECT * FROM station_data_final LIMIT 5
""").df()

,station,delay_rate,delays,total_delay,avg_delay,p50,p95,adjusted_rank,raw_rank,delta,scheduled_trips,ci_low,ci_high,worse_than_baseline
0,VAUGHAN METROPOLITAN CENTRE,1.406221,3029,14119.0,4.661274,3.0,9.0,1,1,0,2154,1.357010,1.456372,True
1,FINCH,1.313849,2922,15398.0,5.269678,4.0,11.0,2,2,0,2224,1.266637,1.361511,True
2,KENNEDY,1.272011,2745,15875.0,5.783242,4.0,14.0,3,3,0,2158,1.224270,1.318825,True
3,KIPLING,1.149420,2477,14549.0,5.873637,4.0,13.0,4,4,0,2155,1.104408,1.194896,True
4,EGLINTON,1.106163,2459,16649.0,6.770638,5.0,15.0,5,5,0,2223,1.061617,1.150697,True


In [140]:
con.sql("""
    CREATE OR REPLACE TABLE rank_comparison AS
        SELECT station, raw_rank AS rank_value, 'raw' AS rank_type FROM station_data_final
        UNION ALL
        SELECT station, adjusted_rank AS rank_value, 'adjusted' AS rank_type FROM station_data_final
""")

con.sql("""
    SELECT * FROM rank_comparison
""").df()

con.sql("""
    COPY rank_comparison TO 'output/rank_comparison.csv' (HEADER, DELIMITER ',')
""")
